# Exportar la revolución sin vivirla: tecnología, IA y crecimiento en Asia-Pacífico


## 2. Datos

### 2.1 Fuentes y variables

El análisis econométrico de **H1** utiliza datos del `World Development Indicators` (WDI) del Banco Mundial, accesibles directamente desde Python via `wbgapi`. Esto garantiza la reproducibilidad completa del análisis.

**Muestra:** 7 países de Asia-Pacífico — China, Malaysia, Vietnam, 
Tailandia, Filipinas, Indonesia y Camboya.  
**Período:** 2010–2023.

| Variable | Rol | Código WDI |
|---|---|---|
| Crecimiento del PIB per cápita (%) | Dependiente | `NY.GDP.PCAP.KD.ZG` |
| Exportaciones de alta tecnología (% exportaciones manufacturadas) | Independiente principal | `TX.VAL.TECH.MF.ZS` |
| Formación bruta de capital fijo (% PIB) | Control | `NE.GDI.FTOT.ZS` |
| Inflación IPC (%) | Control | `FP.CPI.TOTL.ZG` |
| Apertura comercial (X+M / PIB) | Control | `NE.TRD.GNFS.ZS` |

> **Nota metodológica:** H2 y H3 se abordan de forma descriptiva en la 
> sección de discusión, apoyándose en las cifras del informe 
> *EAP Economic Update* (Banco Mundial, abril 2026), dado que no existe 
> una base de datos pública estructurada que permita su contraste 
> econométrico formal.

In [61]:
import plotly.graph_objects as go

# Paleta
COLORS = ['#2c3e50', '#e74c3c', '#27ae60', '#3498db', '#e67e22', '#9b59b6']

# Estilo base reutilizable
BASE_LAYOUT = dict(
    font=dict(family='Georgia, serif', size=12, color='#2c3e50'),
    paper_bgcolor='#fafaf8',
    plot_bgcolor='#fafaf8',
    xaxis=dict(
        gridcolor='#e8e8e4',
        gridwidth=0.5,
        linecolor='#2c3e50',
        showgrid=False,
        tickfont=dict(color='#2c3e50')
    ),
    yaxis=dict(
        gridcolor='#e8e8e4',
        gridwidth=0.8,
        linecolor='#2c3e50',
        tickfont=dict(color='#2c3e50'),
        zeroline=True,
        zerolinecolor='#2c3e50',
        zerolinewidth=1
    ),
    legend=dict(
        bgcolor='#fafaf8',
        bordercolor='#e8e8e4',
        borderwidth=1,
        font=dict(color='#2c3e50')
    ),
    margin=dict(t=110, b=60)
)

def add_title(fig, title, subtitle):
    fig.add_annotation(
        text=f'<b>{title}</b>',
        xref='paper', yref='paper',
        x=0, y=1.15,
        showarrow=False,
        font=dict(size=15, color='#2c3e50', family='Georgia, serif'),
        align='left'
    )
    fig.add_annotation(
        text=subtitle,
        xref='paper', yref='paper',
        x=0, y=1.08,
        showarrow=False,
        font=dict(size=11, color='#888', family='Georgia, serif'),
        align='left'
    )
    fig.add_shape(
        type='line',
        xref='paper', yref='paper',
        x0=0, x1=0.45, y0=1.03, y1=1.03,  # <-- aquí el cambio
        line=dict(color='#2c3e50', width=2)
    )
    )

def add_source(fig, source='World Bank WDI | edalytics.com'):
    fig.add_annotation(
        text=f'Fuente: {source}',
        xref='paper', yref='paper',
        x=0, y=-0.12,
        showarrow=False,
        font=dict(size=10, color='#888'),
        align='left'
    )

SyntaxError: unmatched ')' (2582568583.py, line 59)

In [3]:
import wbgapi as wb
import pandas as pd

#Indicators

indicators: dict[str, str] = {
    'NY.GDP.PCAP.KD.ZG':'gdp_growth',
    'TX.VAL.TECH.MF.ZS':'tech_exports',
    'NE.GDI.FTOT.ZS':'investment',
    'FP.CPI.TOTL.ZG':'inflation',
    'NE.TRD.GNFS.ZS':'trade_openness'
}

countries: list[str] = ['CHN','MYS','VNM', 'THAI', 'PHL', 'IDN', 'KHM']

df = wb.data.DataFrame(
    list(indicators.keys()),
    economy = countries,
    time = range(2010,2024),
    labels = True
).reset_index()

In [10]:
print(df.Country.unique())
print(df.series.unique())

['Cambodia' 'Indonesia' 'Philippines' 'Viet Nam' 'Malaysia' 'China']
['NY.GDP.PCAP.KD.ZG' 'TX.VAL.TECH.MF.ZS' 'NE.GDI.FTOT.ZS' 'FP.CPI.TOTL.ZG'
 'NE.TRD.GNFS.ZS']


In [37]:
df_long = df.melt(
    id_vars = ['economy', 'series', 'Country','Series'],
    var_name = 'year',
    value_name = 'value'
).assign(year = lambda x: x['year'].str.replace('YR','').astype(int)).pivot_table(
    index = ['economy', 'Country', 'year'],
    columns = 'Series',
    values = 'value'
).reset_index()

In [38]:
df_long

Series,economy,Country,year,GDP per capita growth (annual %),Gross fixed capital formation (% of GDP),High-technology exports (% of manufactured exports),"Inflation, consumer prices (annual %)",Trade (% of GDP)
0,CHN,China,2010,10.063424,43.520182,32.150117,3.175325,49.854072
1,CHN,China,2011,8.864807,43.515391,30.500480,5.553899,49.945827
2,CHN,China,2012,7.127013,43.873708,30.861809,2.619524,47.480213
3,CHN,China,2013,7.063225,44.075543,31.585581,2.621050,45.916041
4,CHN,China,2014,6.786670,43.380347,29.703727,1.921642,44.068456
...,...,...,...,...,...,...,...,...
79,VNM,Viet Nam,2019,6.324649,30.362568,40.433430,2.795824,164.704215
80,VNM,Viet Nam,2020,1.915814,30.277818,41.740104,3.220934,163.245863
81,VNM,Viet Nam,2021,1.666516,31.034855,41.539635,1.834716,186.675833
82,VNM,Viet Nam,2022,7.725699,30.565841,42.689179,3.156507,183.153619


In [39]:
df_long.columns.name = None


In [40]:
df_long.head()

,economy,Country,year,GDP per capita growth (annual %),Gross fixed capital formation (% of GDP),High-technology exports (% of manufactured exports),"Inflation, consumer prices (annual %)",Trade (% of GDP)
0,CHN,China,2010,10.063424,43.520182,32.150117,3.175325,49.854072
1,CHN,China,2011,8.864807,43.515391,30.500480,5.553899,49.945827
2,CHN,China,2012,7.127013,43.873708,30.861809,2.619524,47.480213
3,CHN,China,2013,7.063225,44.075543,31.585581,2.621050,45.916041
4,CHN,China,2014,6.786670,43.380347,29.703727,1.921642,44.068456


### 2.4 Renombrado de columnas


In [41]:
df_long = df_long.rename(columns = {
    'GDP per capita growth (annual %)' : 'gdp_growth',
    'Gross fixed capital formation (% of GDP)': 'investment',
    'High-technology exports (% of manufactured exports)': 'tech_exports',
    'Inflation, consumer prices (annual %)': 'inflation',
    'Trade (% of GDP)': 'trade_openness'
})

In [42]:
df_long

,economy,Country,year,gdp_growth,investment,tech_exports,inflation,trade_openness
0,CHN,China,2010,10.063424,43.520182,32.150117,3.175325,49.854072
1,CHN,China,2011,8.864807,43.515391,30.500480,5.553899,49.945827
2,CHN,China,2012,7.127013,43.873708,30.861809,2.619524,47.480213
3,CHN,China,2013,7.063225,44.075543,31.585581,2.621050,45.916041
4,CHN,China,2014,6.786670,43.380347,29.703727,1.921642,44.068456
...,...,...,...,...,...,...,...,...
79,VNM,Viet Nam,2019,6.324649,30.362568,40.433430,2.795824,164.704215
80,VNM,Viet Nam,2020,1.915814,30.277818,41.740104,3.220934,163.245863
81,VNM,Viet Nam,2021,1.666516,31.034855,41.539635,1.834716,186.675833
82,VNM,Viet Nam,2022,7.725699,30.565841,42.689179,3.156507,183.153619


### 2.5 Inspección de valores faltantes


In [43]:
df_long.isnull().sum()

economy           0
Country           0
year              0
gdp_growth        0
investment        0
tech_exports      7
inflation         0
trade_openness    0
dtype: int64

In [44]:
df_long[df_long['tech_exports'].isnull()]

,economy,Country,year,gdp_growth,investment,tech_exports,inflation,trade_openness
56,PHL,Philippines,2010,5.158694,20.402136,NaN,3.789836,66.104279
57,PHL,Philippines,2011,1.837605,18.966639,NaN,4.718417,60.795837
58,PHL,Philippines,2012,4.840765,19.930065,NaN,3.026964,57.842006
59,PHL,Philippines,2013,4.762666,20.783014,NaN,2.582688,55.824781
60,PHL,Philippines,2014,4.615141,20.862173,NaN,3.597823,57.468172
61,PHL,Philippines,2015,4.787251,22.231643,NaN,0.674193,59.141592
62,PHL,Philippines,2016,5.721215,24.996624,NaN,1.253699,61.776066


In [45]:
df_long[df_long['economy'] == 'PHL']['tech_exports']

56          NaN
57          NaN
58          NaN
59          NaN
60          NaN
61          NaN
62          NaN
63    60.317634
64    61.345993
65    62.246675
66    67.045092
67    64.225408
68    66.611092
69    63.977929
Name: tech_exports, dtype: float64

### 2.6 Nota sobre valores faltantes

Filipinas (`PHL`) no reporta datos de exportaciones de alta tecnología 
(`tech_exports`) para el período 2010–2016. A partir de 2017 la serie 
está completa.

Se opta por mantener el **panel desbalanceado** — decisión válida 
econométricamente y consistente con la práctica habitual en datos de panel 
con series incompletas. El estimador de efectos fijos de `linearmodels` 
maneja nativamente observaciones faltantes sin necesidad de imputación.

## 3. Estadísticos descriptivos

### 3.1 Resumen general

In [46]:
df_long.describe().round(2)

,year,gdp_growth,investment,tech_exports,inflation,trade_openness
count,84.00,84.00,84.00,77.00,84.00,84.00
mean,2016.50,4.38,29.75,29.14,3.38,93.04
std,4.06,2.98,6.98,20.82,2.46,47.06
min,2010.00,-10.55,18.21,0.11,-1.14,32.97
25%,2013.00,3.69,24.80,8.45,2.09,47.29
50%,2016.50,4.77,29.84,30.82,2.94,91.11
75%,2020.00,6.08,32.39,47.57,3.89,132.73
max,2023.00,10.06,44.08,67.05,18.68,186.68


### 3.1 Resumen general

```python
df_long.describe().round(2)
```

La muestra cubre 84 observaciones país-año (77 para `tech_exports` por 
los valores faltantes de Filipinas documentados en la sección anterior).

Varios patrones merecen atención:

- **Crecimiento:** media de 4.38% anual, consistente con el dinamismo 
  característico de las economías emergentes de la región. El mínimo 
  de -10.55% corresponde al impacto del COVID-19 en 2020.
- **Exportaciones tecnológicas:** desviación típica de 20.82 puntos — 
  la heterogeneidad entre países es elevada, lo cual favorece la 
  identificación econométrica.
- **Apertura comercial:** rango de 33% a 187% del PIB, reflejo de la 
  diferencia estructural entre China (economía grande y relativamente 
  cerrada) y Vietnam (altamente integrada en cadenas de valor globales).
- **Inflación:** máximo de 18.68% — valor atípico que identificaremos 
  en el análisis por país.

### 3.2 Estadísticos por país

In [48]:
df_long.groupby('Country')[['gdp_growth','investment', 'tech_exports','inflation', 'trade_openness']].mean().round(2)

,gdp_growth,investment,tech_exports,inflation,trade_openness
Country,,,,,
Cambodia,4.60,28.38,2.49,3.15,123.88
China,6.42,42.29,30.33,2.25,40.38
Indonesia,3.64,31.60,9.12,4.20,43.04
Malaysia,2.92,23.12,51.35,2.04,136.89
Philippines,3.77,22.77,63.68,3.45,63.55
Viet Nam,4.93,30.31,35.17,5.16,150.52


### 3.2 Estadísticos por país

```python
df_long.groupby('Country')[['gdp_growth', 'investment', 'tech_exports', 
                             'inflation', 'trade_openness']].mean().round(2)
```

La tabla revela heterogeneidad estructural significativa entre países:

- **Filipinas y Malaysia** lideran en exportaciones tecnológicas (63% y 51% 
  respectivamente) pero registran los crecimientos medios más bajos de la 
  muestra (3.77% y 2.92%) — una primera señal de la paradoja que articula 
  el análisis.
- **China** destaca por una tasa de inversión media del 42% del PIB, 
  reflejo de un modelo de crecimiento históricamente basado en acumulación 
  de capital físico.
- **Vietnam** presenta una apertura comercial del 150% del PIB, confirmando 
  su rol como hub exportador altamente integrado en cadenas de valor 
  regionales.
- **Camboya** muestra el nivel más bajo de exportaciones tecnológicas (2.49%) 
  pero mantiene un crecimiento sólido, sustentado principalmente en 
  manufactura textil y turismo.

In [57]:
countries = df_long['Country'].unique()

fig1 = go.Figure()

for i, country in enumerate(countries):
    d = df_long[df_long['Country'] == country]
    fig1.add_trace(go.Scatter(
        x=d['year'],
        y=d['gdp_growth'],
        name=country,
        mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig1.update_layout(
    **BASE_LAYOUT,
    title=dict(
        text='<b>Crecimiento del PIB per cápita</b><br><sup>Asia-Pacífico, 2010–2023 | % anual</sup>',
        font=dict(size=15, color='#2c3e50'),
        x=0
    ),
    shapes=[dict(
        type='line',
        xref='paper', yref='paper',
        x0=0, x1=1, y0=1.0, y1=1.0,
        line=dict(color='#2c3e50', width=2)
    )]
)

add_source(fig1)
fig1.show()

In [62]:
countries = df_long['Country'].unique()

fig1 = go.Figure()

for i, country in enumerate(countries):
    d = df_long[df_long['Country'] == country]
    fig1.add_trace(go.Scatter(
        x=d['year'],
        y=d['gdp_growth'],
        name=country,
        mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig1.update_layout(**BASE_LAYOUT, title=None)
add_title(fig1, 
          'Crecimiento del PIB per cápita', 
          'Asia-Pacífico, 2010–2023 | % anual')
fig1.update_yaxes(ticksuffix='%')
add_source(fig1)
fig1.show()

El gráfico revela tres patrones estructurales en el crecimiento de la región 
durante el período 2010–2023.

**Convergencia a la baja pre-COVID.** China lidera el crecimiento en 2010 
con un 10.06% pero desacelera de forma sostenida hasta situarse en torno 
al 6% en 2017–2019, convergiendo hacia las tasas del resto de economías 
de la muestra. Esta trayectoria es consistente con la maduración gradual 
de una economía que transita desde el crecimiento extensivo hacia uno 
más intensivo en productividad.

**El shock de 2020.** La pandemia impacta de forma asimétrica: Filipinas 
registra la caída más severa (-10.55%), seguida de Malaysia (-6.71%) e 
Indonesia (-2.89%). Vietnam y Camboya muestran una resiliencia notable — 
Vietnam apenas cae a 1.92%, resultado de su integración en cadenas de 
valor globales que mantuvieron la demanda de exportaciones manufactureras 
incluso durante el confinamiento.

**Recuperación heterogénea post-2021.** Vietnam y Malaysia lideran el 
rebote en 2022 con crecimientos de 7.73%, mientras China se frena 
abruptamente hasta el 3.15% — reflejo de los efectos rezagados de su 
política de cero COVID y la crisis del sector inmobiliario. En 2023 
ningún país recupera los niveles de crecimiento pre-pandemia, lo que 
sugiere un cambio de régimen en la dinámica de crecimiento regional.

In [63]:
fig2 = go.Figure()

for i, country in enumerate(countries):
    d = df_long[df_long['Country'] == country]
    fig2.add_trace(go.Scatter(
        x=d['year'],
        y=d['tech_exports'],
        name=country,
        mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig2.update_layout(**BASE_LAYOUT, title=None)
add_title(fig2,
          'Exportaciones de alta tecnología',
          'Asia-Pacífico, 2010–2023 | % exportaciones manufacturadas')
fig2.update_yaxes(ticksuffix='%')
add_source(fig2)
fig2.show()

El gráfico de exportaciones de alta tecnología expone una de las 
heterogeneidades estructurales más marcadas de la región y constituye 
el punto de partida empírico de la paradoja que articula este análisis.

**Dos clubes tecnológicos.** La muestra se divide con claridad en dos 
grupos. Malaysia y Filipinas operan en una franja alta y estable — 
entre el 47% y el 67% de sus exportaciones manufacturadas son de alta 
tecnología — mientras que Indonesia y Camboya se mantienen por debajo 
del 13% durante prácticamente todo el período. China y Vietnam ocupan 
una posición intermedia en torno al 30–44%.

**Malaysia: liderazgo consolidado con señales de transformación.** 
Malaysia mantiene una intensidad tecnológica exportadora superior al 
47% desde 2010, con una aceleración notable en 2022–2023 hasta alcanzar 
el 59%. Este dinamismo reciente coincide con la oleada de inversión en 
infraestructura de datos e IA documentada en el informe del Banco Mundial.

**Vietnam: la trayectoria más destacada.** Vietnam pasa del 13% en 2010 
al 44% en 2023 — un incremento de 31 puntos porcentuales en trece años, 
resultado directo de su inserción en cadenas de valor globales lideradas 
por Samsung, Intel y LG, que han convertido al país en un hub exportador 
de electrónica de primer orden.

**Camboya: señal emergente a vigilar.** Aunque parte de niveles 
marginales, Camboya registra una aceleración significativa en 2022–2023 
— del 2.98% al 12.99% en apenas dos años — que merece seguimiento en 
análisis futuros.

**La paradoja.** Filipinas lidera en intensidad exportadora tecnológica 
desde 2017 con valores superiores al 60%, pero como vimos en el gráfico 
anterior, registra uno de los crecimientos del PIB per cápita más 
moderados de la muestra. Esta desconexión entre dinamismo exportador y 
crecimiento agregado es precisamente la tensión que el modelo 
econométrico de la siguiente sección busca cuantificar.

In [66]:
print(df_long.pivot_table(index='year', columns='Country', values='tech_exports').round(2).to_string())

Country  Cambodia  China  Indonesia  Malaysia  Philippines  Viet Nam
year                                                                
2010         0.15  32.15      12.31     49.61          NaN     13.13
2011         0.11  30.50      10.82     47.41          NaN     18.81
2012         0.26  30.86      10.87     47.57          NaN     27.16
2013         1.21  31.59       9.68     48.47          NaN     33.60
2014         0.51  29.70       9.33     49.22          NaN     32.05
2015         1.54  30.43       8.89     48.49          NaN     36.37
2016         1.86  30.25       8.00     49.06          NaN     38.06
2017         1.74  30.91       8.45     51.12        60.32     41.74
2018         1.41  31.55       8.21     53.18        61.35     40.75
2019         1.19  30.82       8.09     51.59        62.25     40.43
2020         2.28  31.28       8.42     53.81        67.05     41.74
2021         2.98  30.22       7.20     51.68        64.23     41.54
2022         6.56  27.77       8.3

## 4. Especificación econométrica

### 4.1 Modelo

El análisis empírico de **H1** se basa en un modelo de datos de panel 
con la siguiente especificación:

$$gdp\_growth_{it} = \alpha_i + \beta_1 tech\_exports_{it} + \beta_2 investment_{it} + \beta_3 inflation_{it} + \beta_4 trade\_openness_{it} + \varepsilon_{it}$$

Donde:

- $i$ indexa los países y $t$ los años
- $\alpha_i$ recoge la heterogeneidad no observada específica de cada país 
  — factores estructurales constantes en el tiempo como la calidad 
  institucional, la geografía o el capital humano
- $\varepsilon_{it}$ es el término de error idiosincrático

La elección entre efectos fijos y efectos aleatorios se determina 
empíricamente mediante el **test de Hausman**. La estructura de los 
errores estándar se decide en función de los tests de diagnóstico 
sobre heteroscedasticidad y autocorrelación.

### 4.2 Estimación

In [67]:
from linearmodels.panel import PanelOLS, RandomEffects, compare

In [70]:
df_panel = df_long.set_index(['Country', 'year'])

In [74]:
# Efectos fijos
fe_model = PanelOLS(
    dependent = df_panel['gdp_growth'],
    exog = df_panel[['tech_exports', 'investment', 'inflation', 'trade_openness']],
    entity_effects = True
)

/opt/anaconda3/lib/python3.12/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning:


Inputs contain missing values. Dropping rows with missing observations.



In [75]:
fe_result = fe_model.fit(cov_type = 'clustered', cluster_entity = True)

In [76]:
fe_result

Dep. Variable:,gdp_growth,R-squared:,0.1297
Estimator:,PanelOLS,R-squared (Between):,-0.3193
No. Observations:,77,R-squared (Within):,0.1297
Date:,"Fri, May 15 2026",R-squared (Overall):,-0.1492
Time:,01:12:04,Log-likelihood,-183.98
Cov. Estimator:,Clustered,,
,,F-statistic:,2.4955
Entities:,6,P-value,0.0510
Avg Obs:,12.833,Distribution:,"F(4,67)"
Min Obs:,7.0000,,
Max Obs:,14.000,F-statistic (robust):,13.070


In [77]:
# Efectos aleatorios
re_model = RandomEffects(
    dependent=df_panel['gdp_growth'],
    exog=df_panel[['tech_exports', 'investment', 'inflation', 'trade_openness']]
)

re_result = re_model.fit()

# Test de Hausman
from linearmodels.panel import compare
print(compare({'Efectos Fijos': fe_result, 'Efectos Aleatorios': re_result}))

                     Model Comparison                    
                         Efectos Fijos Efectos Aleatorios
---------------------------------------------------------
Dep. Variable               gdp_growth         gdp_growth
Estimator                     PanelOLS      RandomEffects
No. Observations                    77                 77
Cov. Est.                    Clustered         Unadjusted
R-squared                       0.1297             0.7193
R-Squared (Within)              0.1297             0.0460
R-Squared (Between)            -0.3193             0.9822
R-Squared (Overall)            -0.1492             0.7195
F-statistic                     2.4955             46.767
P-value (F-stat)                0.0510             0.0000
=====================     ============    ===============
tech_exports                   -0.1310            -0.0116
                             (-1.7949)          (-0.7256)
investment                     -0.0434             0.1261
              

/opt/anaconda3/lib/python3.12/site-packages/linearmodels/panel/model.py:2751: MissingValueWarning:


Inputs contain missing values. Dropping rows with missing observations.



In [81]:
import numpy as np
from scipy import stats

# Coeficientes
b_fe = fe_result.params
b_re = re_result.params

# Matrices de covarianza
v_fe = fe_result.cov
v_re = re_result.cov

# Diferencia
b_diff = b_fe - b_re
v_diff = v_fe - v_re

# Estadístico chi-cuadrado
H = float(b_diff.T @ np.linalg.inv(v_diff) @ b_diff)
df = len(b_diff)
p_value = 1 - stats.chi2.cdf(H, df)

print(f'Estadístico Hausman: {H:.4f}')
print(f'Grados de libertad:  {df}')
print(f'P-valor:             {p_value:.4f}')

Estadístico Hausman: 10.1905
Grados de libertad:  4
P-valor:             0.0373


In [82]:
# Residuos del modelo de efectos fijos
residuals = fe_result.resids

# Test de Wooldridge aproximado via autocorrelación de residuos
import pandas as pd

resid_df = residuals.reset_index()
resid_df.columns = ['Country', 'year', 'resid']

# Correlación de residuos con su lag por país
resid_df['resid_lag'] = resid_df.groupby('Country')['resid'].shift(1)

corr = resid_df[['resid', 'resid_lag']].dropna().corr().iloc[0,1]
print(f'Autocorrelación serial de residuos: {corr:.4f}')

Autocorrelación serial de residuos: 0.0575


In [87]:
fe_result

Dep. Variable:,gdp_growth,R-squared:,0.1297
Estimator:,PanelOLS,R-squared (Between):,-0.3193
No. Observations:,77,R-squared (Within):,0.1297
Date:,"Fri, May 15 2026",R-squared (Overall):,-0.1492
Time:,01:12:04,Log-likelihood,-183.98
Cov. Estimator:,Clustered,,
,,F-statistic:,2.4955
Entities:,6,P-value,0.0510
Avg Obs:,12.833,Distribution:,"F(4,67)"
Min Obs:,7.0000,,
Max Obs:,14.000,F-statistic (robust):,13.070


### 4.3 Resultados e interpretación

#### Diagnósticos previos

Antes de interpretar los coeficientes, tres decisiones metodológicas 
quedan respaldadas por los datos:

- **Efectos fijos vs aleatorios:** el test de Hausman arroja un 
  estadístico de 10.19 (p-valor = 0.037), rechazando efectos aleatorios 
  al 5% de significancia. Los efectos individuales están correlacionados 
  con los regresores — usar efectos aleatorios produciría estimadores 
  inconsistentes.
- **Errores estándar:** se utilizan errores clusterizados por país para 
  corregir posible heteroscedasticidad entre entidades.
- **Autocorrelación serial:** la correlación de los residuos con su 
  primer rezago es de 0.057 — prácticamente nula, sin evidencia de 
  correlación serial.

#### Resultados

| Variable | Coeficiente | Std. Error | T-stat | P-valor |
|---|---|---|---|---|
| `tech_exports` | -0.131 | 0.073 | -1.795 | 0.077 |
| `investment` | -0.043 | 0.101 | -0.428 | 0.670 |
| `inflation` | 0.256 | 0.194 | 1.316 | 0.193 |
| `trade_openness` | **0.093** | 0.039 | **2.367** | **0.021** |

*Errores clusterizados por país. R² within = 0.13.*

#### Interpretación

El único regresor estadísticamente significativo al 5% es 
`trade_openness` (p = 0.021). Un incremento de 10 puntos porcentuales 
en la apertura comercial se asocia con un aumento de 0.93 puntos 
porcentuales en el crecimiento del PIB per cápita, manteniendo el resto 
de variables constantes.

`tech_exports` presenta el coeficiente más llamativo del modelo: negativo 
(-0.131) y marginalmente significativo al 10% (p = 0.077). Este resultado, 
contraintuitivo a priori, es coherente con la paradoja documentada en la 
sección descriptiva — una mayor intensidad exportadora tecnológica no se 
traduce automáticamente en mayor crecimiento agregado dentro de los países 
de la muestra una vez controlada la heterogeneidad no observada. Malaysia 
y Filipinas, los dos líderes exportadores, registran los crecimientos más 
moderados del panel.

`investment` e `inflation` no alcanzan significancia estadística en ningún 
umbral convencional.

#### Limitaciones

El R² within de 0.13 indica que el modelo explica una fracción modesta 
de la variación intra-país en el crecimiento. Esto es esperable dado el 
número reducido de unidades (N=6) y sugiere que factores no capturados 
por las variables disponibles — calidad institucional, capital humano, 
política monetaria — tienen un peso relevante en la dinámica de 
crecimiento de la región. Estos resultados deben interpretarse con 
cautela y como punto de partida para análisis más detallados.

In [88]:
# Primera diferencia de todas las variables
diff_vars = ['gdp_growth', 'tech_exports', 'investment', 'inflation', 'trade_openness']

df_diff = (
    df_long
    .sort_values(['Country', 'year'])
    .set_index(['Country', 'year'])
    [diff_vars]
    .groupby(level='Country')
    .diff()
    .dropna()
)

df_diff.head(10)

gdp_growth  tech_exports  investment  inflation  trade_openness
Country  year                                                                 
Cambodia 2011    2.215580     -0.038627    0.098871   1.482052       -0.207895
         2012    0.377639      0.144321    2.289408  -2.544131        6.949728
         2013    0.209345      0.953751    2.225236   0.007309        8.147319
         2014    0.154178     -0.707227    3.214699   0.914064       -0.432644
         2015   -0.758161      1.029532   -0.105622  -2.631757       -2.345312
         2016    0.690075      0.323521    0.502128   1.795208        0.594959
         2017    0.248959     -0.116294    0.268101  -0.106504       -1.549104
         2018    0.839805     -0.333108    0.203731  -0.453550       -2.717796
         2019   -0.849047     -0.215257    0.676658  -0.516510       -2.700976
         2020  -11.546406      1.089053   -0.027952   0.997720        6.368717

In [89]:
from linearmodels.panel import FirstDifferenceOLS

fd_model = FirstDifferenceOLS(
    dependent=df_diff['gdp_growth'],
    exog=df_diff[['tech_exports', 'investment', 'inflation', 'trade_openness']]
)

fd_result = fd_model.fit(cov_type='clustered', cluster_entity=True)
print(fd_result.summary)


                     FirstDifferenceOLS Estimation Summary                      
Dep. Variable:             gdp_growth   R-squared:                        0.4015
Estimator:         FirstDifferenceOLS   R-squared (Between):             -14.538
No. Observations:                  65   R-squared (Within):               0.1467
Date:                Fri, May 15 2026   R-squared (Overall):              0.1151
Time:                        01:30:33   Log-likelihood                   -201.47
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      10.230
Entities:                           6   P-value                           0.0000
Avg Obs:                       11.833   Distribution:                    F(4,61)
Min Obs:                       6.0000                                           
Max Obs:                       13.000   F-statistic (robust):             5.1493
                            

In [90]:
print(compare({
    'Efectos Fijos (niveles)': fe_result,
    'Primeras Diferencias': fd_result
}))

                           Model Comparison                           
                        Efectos Fijos (niveles)   Primeras Diferencias
----------------------------------------------------------------------
Dep. Variable                        gdp_growth             gdp_growth
Estimator                              PanelOLS     FirstDifferenceOLS
No. Observations                             77                     65
Cov. Est.                             Clustered              Clustered
R-squared                                0.1297                 0.4015
R-Squared (Within)                       0.1297                 0.1467
R-Squared (Between)                     -0.3193                -14.538
R-Squared (Overall)                     -0.1492                 0.1151
F-statistic                              2.4955                 10.230
P-value (F-stat)                         0.0510                 0.0000
=====================              ============   ====================
tech_e

In [92]:
er = wb.data.DataFrame(
    'PA.NUS.FCRF',
    economy=['CHN', 'MYS', 'VNM', 'THA', 'PHL', 'IDN', 'KHM'],
    time=range(2010, 2024),
    labels=True
)
print(er.isnull().sum())

Country    0
YR2010     0
YR2011     0
YR2012     0
YR2013     0
YR2014     0
YR2015     0
YR2016     0
YR2017     0
YR2018     0
YR2019     0
YR2020     0
YR2021     0
YR2022     0
YR2023     0
dtype: int64


In [93]:
# Descarga tipo de cambio
er = wb.data.DataFrame(
    'PA.NUS.FCRF',
    economy=['CHN', 'MYS', 'VNM', 'THA', 'PHL', 'IDN', 'KHM'],
    time=range(2010, 2024),
    labels=True
).reset_index()

# Reshape a long
er_long = (
    er.melt(
        id_vars=['economy', 'Country'],
        var_name='year',
        value_name='exchange_rate'
    )
    .assign(year=lambda x: x['year'].str.replace('YR', '').astype(int))
    .sort_values(['Country', 'year'])
)

# Variación porcentual anual
er_long['der'] = er_long.groupby('Country')['exchange_rate'].pct_change() * 100

# Unir al panel principal
df_long = df_long.merge(
    er_long[['economy', 'year', 'der']],
    on=['economy', 'year'],
    how='left'
)

df_long[['Country', 'year', 'der']].head(10)

,Country,year,der
0,China,2010,NaN
1,China,2011,-4.561232
2,China,2012,-2.307969
3,China,2013,-1.846773
4,China,2014,-0.844517
5,China,2015,1.368202
6,China,2016,6.695944
7,China,2017,1.719883
8,China,2018,-2.112784
9,China,2019,4.420038


In [94]:
# Actualizar dataset en diferencias con der
diff_vars = ['gdp_growth', 'tech_exports', 'investment', 
             'inflation', 'trade_openness', 'der']

df_diff = (
    df_long
    .sort_values(['Country', 'year'])
    .set_index(['Country', 'year'])
    [diff_vars]
    .groupby(level='Country')
    .diff()
    .dropna()
)

# Reestimar modelo en primeras diferencias
fd_model2 = FirstDifferenceOLS(
    dependent=df_diff['gdp_growth'],
    exog=df_diff[['tech_exports', 'investment', 
                  'inflation', 'trade_openness', 'der']]
)

fd_result2 = fd_model2.fit(cov_type='clustered', cluster_entity=True)
print(fd_result2.summary)

                     FirstDifferenceOLS Estimation Summary                      
Dep. Variable:             gdp_growth   R-squared:                        0.4270
Estimator:         FirstDifferenceOLS   R-squared (Between):             -25.220
No. Observations:                  60   R-squared (Within):               0.1895
Date:                Fri, May 15 2026   R-squared (Overall):              0.1529
Time:                        01:35:35   Log-likelihood                   -186.98
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      8.1960
Entities:                           6   P-value                           0.0000
Avg Obs:                       11.000   Distribution:                    F(5,55)
Min Obs:                       6.0000                                           
Max Obs:                       12.000   F-statistic (robust):             9.9442
                            

In [95]:
# Modelo parsimonioso final
fd_model_final = FirstDifferenceOLS(
    dependent=df_diff['gdp_growth'],
    exog=df_diff[['tech_exports', 'investment', 'trade_openness']]
)

fd_result_final = fd_model_final.fit(cov_type='clustered', cluster_entity=True)
print(fd_result_final.summary)

                     FirstDifferenceOLS Estimation Summary                      
Dep. Variable:             gdp_growth   R-squared:                        0.4122
Estimator:         FirstDifferenceOLS   R-squared (Between):             -24.764
No. Observations:                  60   R-squared (Within):               0.1825
Date:                Fri, May 15 2026   R-squared (Overall):              0.1468
Time:                        01:36:21   Log-likelihood                   -187.74
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      13.322
Entities:                           6   P-value                           0.0000
Avg Obs:                       11.000   Distribution:                    F(3,57)
Min Obs:                       6.0000                                           
Max Obs:                       12.000   F-statistic (robust):             3.8479
                            

In [96]:
print(compare({
    'FE Niveles'        : fe_result,
    'FD Completo'       : fd_result2,
    'FD Parsimonioso'   : fd_result_final
}))

                                  Model Comparison                                  
                            FE Niveles            FD Completo        FD Parsimonioso
------------------------------------------------------------------------------------
Dep. Variable               gdp_growth             gdp_growth             gdp_growth
Estimator                     PanelOLS     FirstDifferenceOLS     FirstDifferenceOLS
No. Observations                    77                     60                     60
Cov. Est.                    Clustered              Clustered              Clustered
R-squared                       0.1297                 0.4270                 0.4122
R-Squared (Within)              0.1297                 0.1895                 0.1825
R-Squared (Between)            -0.3193                -25.220                -24.764
R-Squared (Overall)            -0.1492                 0.1529                 0.1468
F-statistic                     2.4955                 8.1960    

In [97]:
print(fd_result_final.params)
print(fd_result_final.std_errors)
print(fd_result_final.conf_int())

tech_exports     -0.868197
investment        0.729867
trade_openness    0.319530
Name: parameter, dtype: float64
tech_exports      0.319419
investment        0.576799
trade_openness    0.128603
Name: std_error, dtype: float64
                   lower     upper
tech_exports   -1.507823 -0.228570
investment     -0.425154  1.884888
trade_openness  0.062006  0.577054


In [99]:
import pandas as pd

# Datos del modelo parsimonioso
coef_df = pd.DataFrame({
    'coef'  : fd_result_final.params,
    'lower' : fd_result_final.conf_int()['lower'],
    'upper' : fd_result_final.conf_int()['upper']
}).reset_index()
coef_df.columns = ['variable', 'coef', 'lower', 'upper']

# Etiquetas legibles
labels = {
    'tech_exports'  : 'Exportaciones tecnológicas',
    'investment'    : 'Inversión',
    'trade_openness': 'Apertura comercial'
}
coef_df['variable'] = coef_df['variable'].map(labels)

# Color según significancia
coef_df['color'] = coef_df.apply(
    lambda r: '#e74c3c' if r['lower'] > 0 or r['upper'] < 0 else '#95a5a6', 
    axis=1
)

fig_coef = go.Figure()

for _, row in coef_df.iterrows():
    # Intervalo de confianza
    fig_coef.add_trace(go.Scatter(
        x=[row['lower'], row['upper']],
        y=[row['variable'], row['variable']],
        mode='lines',
        line=dict(color=row['color'], width=2),
        showlegend=False
    ))
    # Punto central
    fig_coef.add_trace(go.Scatter(
        x=[row['coef']],
        y=[row['variable']],
        mode='markers',
        marker=dict(color=row['color'], size=10),
        showlegend=False
    ))

# Línea vertical en cero
fig_coef.update_layout(**BASE_LAYOUT, title=None)
fig_coef.update_xaxes(title='Coeficiente')
add_title(fig_coef,
          'Coeficientes del modelo parsimonioso',
          'Intervalos de confianza al 95% | Errores clusterizados por país')
add_source(fig_coef, 'World Bank WDI | Estimación propia')
fig_coef.show()

In [101]:
print(df_long[['Country', 'year', 'tech_exports', 'gdp_growth']].dropna().to_string())

        Country  year  tech_exports  gdp_growth
0         China  2010     32.150117   10.063424
1         China  2011     30.500480    8.864807
2         China  2012     30.861809    7.127013
3         China  2013     31.585581    7.063225
4         China  2014     29.703727    6.786670
5         China  2015     30.431267    6.358550
6         China  2016     30.254523    6.165427
7         China  2017     30.911864    6.246266
8         China  2018     31.548148    6.258612
9         China  2019     30.818630    5.692901
10        China  2020     31.276615    2.096867
11        China  2021     30.219073    8.473227
12        China  2022     27.765061    3.147700
13        China  2023     26.568214    5.524315
14    Indonesia  2010     12.311815    4.893262
15    Indonesia  2011     10.817141    4.822943
16    Indonesia  2012     10.871757    4.675404
17    Indonesia  2013      9.676063    4.256039
18    Indonesia  2014      9.328508    3.779686
19    Indonesia  2015      8.889878    3

In [103]:
# Unir diferencias con nombres de países
df_scatter = df_diff.reset_index().merge(
    df_long[['Country', 'economy', 'year']],
    on=['Country', 'year'],
    how='left'
).dropna(subset=['tech_exports', 'gdp_growth'])

fig_scatter = go.Figure()

for i, country in enumerate(df_scatter['Country'].unique()):
    d = df_scatter[df_scatter['Country'] == country]
    fig_scatter.add_trace(go.Scatter(
        x=d['tech_exports'],
        y=d['gdp_growth'],
        mode='markers',
        name=country,
        marker=dict(color=COLORS[i], size=8, opacity=0.8)
    ))

# Línea de tendencia global
x_vals = df_scatter['tech_exports']
y_vals = df_scatter['gdp_growth']
z = np.polyfit(x_vals, y_vals, 1)
p = np.poly1d(z)
x_line = np.linspace(x_vals.min(), x_vals.max(), 100)

fig_scatter.add_trace(go.Scatter(
    x=x_line,
    y=p(x_line),
    mode='lines',
    name='Tendencia',
    line=dict(color='#2c3e50', width=1.5, dash='dash'),
    showlegend=True
))

fig_scatter.update_layout(**BASE_LAYOUT, title=None)
fig_scatter.update_xaxes(title='Δ Exportaciones tecnológicas (pp)')
fig_scatter.update_yaxes(title='Crecimiento PIB per cápita (%)', ticksuffix='%')
add_title(fig_scatter,
          'Variación en exportaciones tecnológicas y crecimiento',
          'Países EAP, 2011–2023 | Cada punto es un país-año')
add_source(fig_scatter, 'World Bank WDI | Estimación propia')

fig_scatter.show()

In [104]:
fig3 = go.Figure()

for i, country in enumerate(countries):
    d = df_long[df_long['Country'] == country]
    fig3.add_trace(go.Scatter(
        x=d['year'],
        y=d['trade_openness'],
        name=country,
        mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig3.update_layout(**BASE_LAYOUT, title=None)
fig3.update_yaxes(ticksuffix='%')
add_title(fig3,
          'Apertura comercial',
          'Asia-Pacífico, 2010–2023 | (X+M)/PIB en %')
add_source(fig3)

print(df_long.pivot_table(index='year', columns='Country', values='trade_openness').round(2).to_string())
fig3.show()

Country  Cambodia  China  Indonesia  Malaysia  Philippines  Viet Nam
year                                                                
2010       110.00  49.85      46.70    157.94        66.10    113.98
2011       109.79  49.95      50.18    154.94        60.80    125.26
2012       116.74  47.48      49.58    147.84        57.84    123.22
2013       124.89  45.92      48.64    142.72        55.82    130.85
2014       124.45  44.07      48.08    138.31        57.47    135.41
2015       122.11  38.70      41.94    131.37        59.14    144.91
2016       122.70  36.18      37.42    126.90        61.78    145.41
2017       121.16  36.95      39.36    133.16        68.17    160.98
2018       118.44  36.89      43.07    130.40        72.16    164.66
2019       115.74  35.20      37.63    123.03        68.84    164.70
2020       122.11  34.04      32.97    116.79        58.17    163.25
2021       146.18  36.52      40.20    134.04        63.48    186.68
2022       145.87  37.44      45.4

La evolución de la apertura comercial revela dos modelos estructurales 
claramente diferenciados dentro de la región.

**El grupo hiperintegrado.** Vietnam, Malaysia y Cambodia superan el 
100% del PIB en comercio exterior durante todo el período — en el caso 
de Vietnam alcanza el 187% en 2021, máximo histórico de la muestra. 
Estas economías pequeñas y altamente especializadas funcionan 
esencialmente como nodos de tránsito en cadenas de valor globales, 
donde el valor del comercio exterior supera ampliamente el tamaño de 
su economía doméstica.

**El grupo de economías grandes y relativamente cerradas.** China e 
Indonesia muestran una tendencia secular a la baja en apertura comercial 
— China pasa del 49.9% en 2011 al 34% en 2020, reflejo de su transición 
hacia un modelo de crecimiento más orientado al consumo interno. Indonesia 
sigue una trayectoria similar, con apertura comercial que oscila entre 
el 33% y el 50% a lo largo del período.

**El shock de 2020 y la recuperación asimétrica.** La pandemia contrae 
la apertura comercial en todos los países, pero la recuperación es 
notablemente asimétrica. Vietnam y Cambodia aceleran hasta nuevos máximos 
en 2021 — impulsados por la demanda global de electrónica y manufactura 
ligera — mientras China e Indonesia recuperan apenas los niveles 
pre-pandemia.

Este resultado refuerza el coeficiente positivo y robusto de 
`trade_openness` en el modelo econométrico: la integración comercial 
es el canal más consistente de transmisión al crecimiento en la región, 
por encima de la intensidad exportadora tecnológica.

In [105]:
import numpy as np

# Matriz de correlación
corr_matrix = df_long[['gdp_growth', 'tech_exports', 'investment', 
                        'inflation', 'trade_openness', 'der']].corr().round(2)

fig_heat = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.columns.tolist(),
    colorscale=[
        [0.0, '#e74c3c'],
        [0.5, '#fafaf8'],
        [1.0, '#2c3e50']
    ],
    zmid=0,
    text=corr_matrix.values,
    texttemplate='%{text}',
    textfont=dict(size=11, family='Georgia, serif'),
    showscale=True
))

fig_heat.update_layout(**BASE_LAYOUT, title=None)
add_title(fig_heat,
          'Matriz de correlaciones',
          'Variables del panel | Pearson')
add_source(fig_heat, 'World Bank WDI | Estimación propia')

print(corr_matrix.to_string())
fig_heat.show()

                gdp_growth  tech_exports  investment  inflation  trade_openness   der
gdp_growth            1.00         -0.13        0.33       0.14           -0.05  0.03
tech_exports         -0.13          1.00       -0.30      -0.17            0.22  0.04
investment            0.33         -0.30        1.00      -0.02           -0.42 -0.04
inflation             0.14         -0.17       -0.02       1.00            0.02  0.28
trade_openness       -0.05          0.22       -0.42       0.02            1.00 -0.06
der                   0.03          0.04       -0.04       0.28           -0.06  1.00


La matriz de correlaciones ofrece una primera aproximación a las 
relaciones bivariadas entre las variables del panel, previa a la 
estimación econométrica.

**`investment` es el regresor más correlacionado con el crecimiento** 
(r = 0.33), lo que contrasta con su falta de significancia en el modelo 
de primeras diferencias. Esta aparente contradicción sugiere que la 
correlación positiva en niveles refleja heterogeneidad entre países — 
economías que sistemáticamente invierten más también crecen más — pero 
una vez eliminada esa heterogeneidad mediante diferenciación, el efecto 
intra-país de la inversión sobre el crecimiento no es estadísticamente 
distinguible de cero con N=6.

**`tech_exports` correlaciona negativamente con el crecimiento** 
(r = -0.13) y con la inversión (r = -0.30), pero positivamente con 
la apertura comercial (r = 0.22) — señal de que los países más 
integrados comercialmente tienden a especializarse en exportaciones 
tecnológicas, pero esa especialización no se traduce en mayor inversión 
doméstica.

**La correlación más elevada del panel es entre `investment` y 
`trade_openness`** (r = -0.42) — relación negativa que refleja la 
diferencia estructural entre China (alta inversión, baja apertura) 
y Vietnam o Cambodia (baja inversión, alta apertura). Esta correlación 
moderada entre regresores no compromete la estimación pero es 
consistente con la inestabilidad del coeficiente de `investment` 
entre especificaciones.

**`der` e `inflation`** muestran correlaciones bajas con todas las 
variables, confirmando su escaso poder explicativo en el modelo 
y justificando su exclusión de la especificación final.

In [106]:
# Fitted values y residuos del modelo parsimonioso
fitted = fd_result_final.fitted_values
resids = fd_result_final.resids

fig_diag = go.Figure()

fig_diag.add_trace(go.Scatter(
    x=fitted,
    y=resids,
    mode='markers',
    marker=dict(color='#2c3e50', size=7, opacity=0.7),
    showlegend=False
))

# Línea horizontal en cero
fig_diag.add_hline(y=0, line=dict(color='#e74c3c', width=1.5, dash='dash'))

fig_diag.update_layout(**BASE_LAYOUT, title=None)
fig_diag.update_xaxes(title='Valores ajustados')
fig_diag.update_yaxes(title='Residuos')
add_title(fig_diag,
          'Diagnóstico del modelo: valores ajustados vs residuos',
          'Modelo parsimonioso en primeras diferencias')
add_source(fig_diag, 'Estimación propia')

print(pd.DataFrame({
    'fitted': fitted.values,
    'resids': resids.values
}).describe().round(3).to_string())

fig_diag.show()

ValueError: Per-column arrays must each be 1-dimensional

In [107]:
# Dummy política industrial activa
# China y Malaysia clasificados como política industrial activa
policy_countries = ['China', 'Malaysia']

df_long['policy'] = df_long['Country'].apply(
    lambda x: 1 if x in policy_countries else 0
)

# Término de interacción
df_long['tech_policy'] = df_long['tech_exports'] * df_long['policy']

df_long[['Country', 'year', 'policy', 'tech_exports', 'tech_policy']].head(10)

,Country,year,policy,tech_exports,tech_policy
0,China,2010,1,32.150117,32.150117
1,China,2011,1,30.500480,30.500480
2,China,2012,1,30.861809,30.861809
3,China,2013,1,31.585581,31.585581
4,China,2014,1,29.703727,29.703727
5,China,2015,1,30.431267,30.431267
6,China,2016,1,30.254523,30.254523
7,China,2017,1,30.911864,30.911864
8,China,2018,1,31.548148,31.548148
9,China,2019,1,30.818630,30.818630


In [108]:
# Actualizar diferencias con interacción
diff_vars = ['gdp_growth', 'tech_exports', 'investment', 
             'trade_openness', 'tech_policy']

df_diff_d = (
    df_long
    .sort_values(['Country', 'year'])
    .set_index(['Country', 'year'])
    [diff_vars]
    .groupby(level='Country')
    .diff()
    .dropna()
)

# Estimar modelo con interacción
fd_model_d = FirstDifferenceOLS(
    dependent=df_diff_d['gdp_growth'],
    exog=df_diff_d[['tech_exports', 'investment', 
                    'trade_openness', 'tech_policy']]
)

fd_result_d = fd_model_d.fit(cov_type='clustered', cluster_entity=True)
print(fd_result_d.summary)

                     FirstDifferenceOLS Estimation Summary                      
Dep. Variable:             gdp_growth   R-squared:                        0.4184
Estimator:         FirstDifferenceOLS   R-squared (Between):             -19.833
No. Observations:                  65   R-squared (Within):               0.1236
Date:                Fri, May 15 2026   R-squared (Overall):              0.0802
Time:                        03:37:37   Log-likelihood                   -200.54
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      10.969
Entities:                           6   P-value                           0.0000
Avg Obs:                       11.833   Distribution:                    F(4,61)
Min Obs:                       6.0000                                           
Max Obs:                       13.000   F-statistic (robust):             2.3573
                            

In [109]:
from linearmodels.panel import BetweenOLS
from linearmodels.panel import PanelOLS

# Modelo dinámico con lag de la dependiente
df_long_sorted = df_long.sort_values(['Country', 'year'])
df_long_sorted['gdp_growth_lag'] = df_long_sorted.groupby('Country')['gdp_growth'].shift(1)

df_panel_dyn = df_long_sorted.dropna(subset=['gdp_growth_lag']).set_index(['Country', 'year'])

fe_dynamic = PanelOLS(
    dependent=df_panel_dyn['gdp_growth'],
    exog=df_panel_dyn[['gdp_growth_lag', 'tech_exports', 
                        'investment', 'trade_openness']],
    entity_effects=True
)

fe_dynamic_result = fe_dynamic.fit(cov_type='clustered', cluster_entity=True)
print(fe_dynamic_result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:             gdp_growth   R-squared:                        0.0971
Estimator:                   PanelOLS   R-squared (Between):             -2.5233
No. Observations:                  72   R-squared (Within):               0.0971
Date:                Fri, May 15 2026   R-squared (Overall):             -1.3102
Time:                        03:38:27   Log-likelihood                   -174.33
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      1.6668
Entities:                           6   P-value                           0.1691
Avg Obs:                       12.000   Distribution:                    F(4,62)
Min Obs:                       7.0000                                           
Max Obs:                       13.000   F-statistic (robust):             9.6316
                            

/opt/anaconda3/lib/python3.12/site-packages/linearmodels/panel/model.py:1258: MissingValueWarning:


Inputs contain missing values. Dropping rows with missing observations.



In [110]:
# Dummy umbral en 30% de tech_exports
df_long['high_tech'] = (df_long['tech_exports'] >= 30).astype(int)

# Interacción
df_long['tech_high'] = df_long['tech_exports'] * df_long['high_tech']

# Actualizar diferencias
diff_vars_b = ['gdp_growth', 'tech_exports', 'investment',
               'trade_openness', 'tech_high']

df_diff_b = (
    df_long
    .sort_values(['Country', 'year'])
    .set_index(['Country', 'year'])
    [diff_vars_b]
    .groupby(level='Country')
    .diff()
    .dropna()
)

fd_model_b = FirstDifferenceOLS(
    dependent=df_diff_b['gdp_growth'],
    exog=df_diff_b[['tech_exports', 'investment',
                    'trade_openness', 'tech_high']]
)

fd_result_b = fd_model_b.fit(cov_type='clustered', cluster_entity=True)
print(fd_result_b.summary)

                     FirstDifferenceOLS Estimation Summary                      
Dep. Variable:             gdp_growth   R-squared:                        0.4301
Estimator:         FirstDifferenceOLS   R-squared (Between):             -15.359
No. Observations:                  65   R-squared (Within):               0.1424
Date:                Fri, May 15 2026   R-squared (Overall):              0.1085
Time:                        03:41:49   Log-likelihood                   -199.87
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      11.510
Entities:                           6   P-value                           0.0000
Avg Obs:                       11.833   Distribution:                    F(4,61)
Min Obs:                       6.0000                                           
Max Obs:                       13.000   F-statistic (robust):             2.4532
                            

In [111]:
print(compare({
    'FE Niveles'        : fe_result,
    'FD Parsimonioso'   : fd_result_final,
    'FD Interacción'    : fd_result_d,
    'FD Umbral'         : fd_result_b
}))

                                              Model Comparison                                             
                            FE Niveles        FD Parsimonioso         FD Interacción              FD Umbral
-----------------------------------------------------------------------------------------------------------
Dep. Variable               gdp_growth             gdp_growth             gdp_growth             gdp_growth
Estimator                     PanelOLS     FirstDifferenceOLS     FirstDifferenceOLS     FirstDifferenceOLS
No. Observations                    77                     60                     65                     65
Cov. Est.                    Clustered              Clustered              Clustered              Clustered
R-squared                       0.1297                 0.4122                 0.4184                 0.4301
R-Squared (Within)              0.1297                 0.1825                 0.1236                 0.1424
R-Squared (Between)         

In [112]:
rd = wb.data.DataFrame(
    'GB.XPD.RSDV.GD.ZS',
    economy=['CHN', 'MYS', 'VNM', 'THA', 'PHL', 'IDN', 'KHM'],
    time=range(2010, 2024),
    labels=True
)
print(rd.isnull().sum())

Country    0
YR2010     5
YR2011     2
YR2012     5
YR2013     2
YR2014     4
YR2015     1
YR2016     3
YR2017     3
YR2018     2
YR2019     2
YR2020     2
YR2021     4
YR2022     4
YR2023     4
dtype: int64


In [116]:
patents = wb.data.DataFrame(
    'IP.PAT.RESD',
    economy=['CHN', 'MYS', 'VNM', 'THA', 'PHL', 'IDN', 'KHM'],
    time=range(2010, 2024),
    labels=True
)
print(patents.isnull().sum())

Country    0
YR2010     1
YR2011     1
YR2012     2
YR2013     0
YR2014     0
YR2015     1
YR2016     0
YR2017     1
YR2018     0
YR2019     1
YR2020     1
YR2021     1
YR2022     7
YR2023     7
dtype: int64


In [117]:
print(patents[patents.isnull().any(axis=1)][['Country']])

             Country
economy             
KHM         Cambodia
IDN        Indonesia
PHL      Philippines
THA         Thailand
VNM         Viet Nam
MYS         Malaysia
CHN            China


In [118]:
print(patents.set_index('Country').T.to_string())

Country  Cambodia  Indonesia  Philippines  Thailand  Viet Nam  Malaysia      China
YR2010        NaN      508.0        170.0    1214.0     306.0    1231.0   293066.0
YR2011        NaN      533.0        186.0     927.0     300.0    1076.0   415829.0
YR2012        NaN        NaN        162.0    1020.0     382.0    1114.0   535313.0
YR2013        1.0      663.0        220.0    1572.0     443.0    1199.0   704936.0
YR2014        2.0      702.0        334.0    1006.0     487.0    1353.0   801135.0
YR2015        NaN     1058.0        375.0    1029.0     582.0    1272.0   968252.0
YR2016        3.0     1101.0        327.0    1098.0     560.0    1109.0  1204981.0
YR2017        NaN     2271.0        323.0     979.0     592.0    1166.0  1245709.0
YR2018        1.0     1407.0        529.0     904.0     646.0    1116.0  1393815.0
YR2019        NaN     3093.0        501.0     865.0     720.0    1071.0  1243568.0
YR2020        NaN     1309.0        476.0     863.0    1021.0     989.0  1344817.0
YR20

In [121]:
print(patents.index)
print(patents.columns)

Index(['KHM', 'IDN', 'PHL', 'THA', 'VNM', 'MYS', 'CHN'], dtype='object', name='economy')
Index(['Country', 'YR2010', 'YR2011', 'YR2012', 'YR2013', 'YR2014', 'YR2015',
       'YR2016', 'YR2017', 'YR2018', 'YR2019', 'YR2020', 'YR2021', 'YR2022',
       'YR2023'],
      dtype='object')


In [122]:
patents_long = (
    patents
    .reset_index()
    .melt(id_vars=['economy', 'Country'],
          var_name='year',
          value_name='patents')
    .assign(year=lambda x: x['year'].str.replace('YR', '').astype(int))
    .query('year <= 2021')
    .query('Country != "Cambodia"')
    .dropna(subset=['patents'])
)

patents_long['log_patents'] = np.log(patents_long['patents'])

df_long = df_long.merge(
    patents_long[['economy', 'year', 'log_patents']],
    on=['economy', 'year'],
    how='left'
)

print(df_long[['Country', 'year', 'tech_exports', 'log_patents']].dropna().to_string())

        Country  year  tech_exports  log_patents
0         China  2010     32.150117    12.588153
1         China  2011     30.500480    12.938029
2         China  2012     30.861809    13.190607
3         China  2013     31.585581    13.465862
4         China  2014     29.703727    13.593785
5         China  2015     30.431267    13.783248
6         China  2016     30.254523    14.001974
7         China  2017     30.911864    14.035215
8         China  2018     31.548148    14.147555
9         China  2019     30.818630    14.033495
10        China  2020     31.276615    14.111769
11        China  2021     30.219073    14.170835
14    Indonesia  2010     12.311815     6.230481
15    Indonesia  2011     10.817141     6.278521
17    Indonesia  2013      9.676063     6.496775
18    Indonesia  2014      9.328508     6.553933
19    Indonesia  2015      8.889878     6.964136
20    Indonesia  2016      7.998610     7.003974
21    Indonesia  2017      8.446327     7.727976
22    Indonesia  201

In [123]:
# Panel con patentes
df_panel_pat = (
    df_long
    .dropna(subset=['tech_exports', 'log_patents'])
    .set_index(['Country', 'year'])
)

fe_patents = PanelOLS(
    dependent=df_panel_pat['log_patents'],
    exog=df_panel_pat[['tech_exports']],
    entity_effects=True
)

fe_patents_result = fe_patents.fit(cov_type='clustered', cluster_entity=True)
print(fe_patents_result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:            log_patents   R-squared:                        0.0928
Estimator:                   PanelOLS   R-squared (Between):              0.1945
No. Observations:                  52   R-squared (Within):               0.0928
Date:                Fri, May 15 2026   R-squared (Overall):              0.1777
Time:                        06:41:42   Log-likelihood                   -24.169
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      4.7055
Entities:                           5   P-value                           0.0353
Avg Obs:                       10.400   Distribution:                    F(1,46)
Min Obs:                       5.0000                                           
Max Obs:                       12.000   F-statistic (robust):             4.8736
                            

In [124]:
# Media por país de ambas variables
df_critic = (
    df_long
    .dropna(subset=['tech_exports', 'log_patents'])
    .groupby('Country')[['tech_exports', 'log_patents']]
    .mean()
    .reset_index()
)

fig_critic = go.Figure()

for i, row in df_critic.iterrows():
    fig_critic.add_trace(go.Scatter(
        x=[row['tech_exports']],
        y=[row['log_patents']],
        mode='markers+text',
        name=row['Country'],
        text=[row['Country']],
        textposition='top center',
        marker=dict(color=COLORS[i], size=12),
        showlegend=False
    ))

fig_critic.update_layout(**BASE_LAYOUT, title=None)
fig_critic.update_xaxes(title='Exportaciones tecnológicas (% exp. manufacturadas)')
fig_critic.update_yaxes(title='Log patentes residentes')
add_title(fig_critic,
          '¿Exportar tecnología equivale a innovar?',
          'Media 2010–2021 | Cada punto es un país')
add_source(fig_critic, 'World Bank WDI | Estimación propia')

print(df_critic.to_string())
fig_critic.show()

       Country  tech_exports  log_patents
0        China     30.855153    13.671711
1    Indonesia      9.036311     6.996455
2     Malaysia     50.101392     7.025533
3  Philippines     63.036160     6.125014
4     Viet Nam     33.782184     6.306899


In [125]:
# Media por país de ambas variables
df_critic = (
    df_long
    .dropna(subset=['tech_exports', 'log_patents'])
    .groupby('Country')[['tech_exports', 'log_patents']]
    .mean()
    .reset_index()
)

fig_critic = go.Figure()

for i, row in df_critic.iterrows():
    fig_critic.add_trace(go.Scatter(
        x=[row['tech_exports']],
        y=[row['log_patents']],
        mode='markers+text',
        name=row['Country'],
        text=[row['Country']],
        textposition='top center',
        marker=dict(color=COLORS[i], size=12),
        showlegend=False
    ))

fig_critic.update_layout(**BASE_LAYOUT, title=None)
fig_critic.update_xaxes(title='Exportaciones tecnológicas (% exp. manufacturadas)')
fig_critic.update_yaxes(title='Log patentes residentes')
add_title(fig_critic,
          '¿Exportar tecnología equivale a innovar?',
          'Media 2010–2021 | Cada punto es un país')
add_source(fig_critic, 'World Bank WDI | Estimación propia')

print(df_critic.to_string())
fig_critic.show()

       Country  tech_exports  log_patents
0        China     30.855153    13.671711
1    Indonesia      9.036311     6.996455
2     Malaysia     50.101392     7.025533
3  Philippines     63.036160     6.125014
4     Viet Nam     33.782184     6.306899


In [126]:
df_critic = (
    df_long
    .dropna(subset=['tech_exports', 'log_patents'])
    .query('Country != "China"')
    .groupby('Country')[['tech_exports', 'log_patents']]
    .mean()
    .reset_index()
)

fig_critic = go.Figure()

for i, row in df_critic.iterrows():
    fig_critic.add_trace(go.Scatter(
        x=[row['tech_exports']],
        y=[row['log_patents']],
        mode='markers+text',
        name=row['Country'],
        text=[row['Country']],
        textposition='top center',
        marker=dict(color=COLORS[i], size=12),
        showlegend=False
    ))

fig_critic.update_layout(**BASE_LAYOUT, title=None)
fig_critic.update_xaxes(title='Exportaciones tecnológicas (% exp. manufacturadas)')
fig_critic.update_yaxes(title='Log patentes residentes')
add_title(fig_critic,
          '¿Exportar tecnología equivale a innovar?',
          'Media 2010–2021 | Excluye China')
add_source(fig_critic, 'World Bank WDI | Estimación propia')

print(df_critic.to_string())
fig_critic.show()

       Country  tech_exports  log_patents
0    Indonesia      9.036311     6.996455
1     Malaysia     50.101392     7.025533
2  Philippines     63.036160     6.125014
3     Viet Nam     33.782184     6.306899


In [127]:
df_critic = (
    df_long
    .query('Country != "China"')
    .dropna(subset=['tech_exports', 'log_patents'])
    .groupby('Country')[['tech_exports', 'log_patents']]
    .mean()
    .reset_index()
)

print(df_critic.to_string())

       Country  tech_exports  log_patents
0    Indonesia      9.036311     6.996455
1     Malaysia     50.101392     7.025533
2  Philippines     63.036160     6.125014
3     Viet Nam     33.782184     6.306899


In [128]:
fig_critic = go.Figure()

for i, row in df_critic.iterrows():
    fig_critic.add_trace(go.Scatter(
        x=[row['tech_exports']],
        y=[row['log_patents']],
        mode='markers+text',
        name=row['Country'],
        text=[row['Country']],
        textposition='top center',
        marker=dict(color=COLORS[i], size=12),
        showlegend=False
    ))

fig_critic.update_layout(**BASE_LAYOUT, title=None)
fig_critic.update_xaxes(title='Exportaciones tecnológicas (% exp. manufacturadas)')
fig_critic.update_yaxes(title='Log patentes residentes')
add_title(fig_critic,
          '¿Exportar tecnología equivale a innovar?',
          'Media 2010–2021 | Excluye China')
add_source(fig_critic, 'World Bank WDI | Estimación propia')

print(df_critic.to_string())
fig_critic.show()

       Country  tech_exports  log_patents
0    Indonesia      9.036311     6.996455
1     Malaysia     50.101392     7.025533
2  Philippines     63.036160     6.125014
3     Viet Nam     33.782184     6.306899


In [129]:
fig_critic = go.Figure()

for i, row in df_critic.iterrows():
    fig_critic.add_trace(go.Scatter(
        x=[row['tech_exports']],
        y=[row['log_patents']],
        mode='markers+text',
        name=row['Country'],
        text=[row['Country']],
        textposition='top center',
        marker=dict(color=COLORS[i], size=12),
        showlegend=False
    ))

fig_critic.update_layout(**BASE_LAYOUT, title=None)
fig_critic.update_xaxes(title='Exportaciones tecnológicas (% exp. manufacturadas)')
fig_critic.update_yaxes(title='Log patentes residentes')
add_title(fig_critic,
          '¿Exportar tecnología equivale a innovar?',
          'Media 2010–2021 | Excluye China')
add_source(fig_critic, 'World Bank WDI | Estimación propia')

print(df_critic.to_string())
fig_critic.show()

       Country  tech_exports  log_patents
0    Indonesia      9.036311     6.996455
1     Malaysia     50.101392     7.025533
2  Philippines     63.036160     6.125014
3     Viet Nam     33.782184     6.306899


In [130]:
# Enterprise Survey está en WDI como indicadores agregados
enterprise = wb.data.DataFrame(
    'IC.FRM.DURS',  # Días para obtener electricidad -- proxy de entorno empresarial
    economy=['CHN', 'MYS', 'VNM', 'THA', 'PHL', 'IDN', 'KHM'],
    time=range(2010, 2024),
    labels=True
)
print(enterprise.isnull().sum())

Country    0
YR2010     7
YR2011     7
YR2012     6
YR2013     7
YR2014     7
YR2015     3
YR2016     5
YR2017     7
YR2018     7
YR2019     6
YR2020     7
YR2021     7
YR2022     7
YR2023     3
dtype: int64


In [131]:
import requests

# Test API OpenDOSM
url = "https://api.data.gov.my/data-catalogue?id=gdp_qtr_nominal&limit=3"
response = requests.get(url=url)
print(f"Status: {response.status_code}")
print(response.json())

Status: 200
[{'date': '2015-01-01', 'value': 281643.0, 'series': 'abs'}, {'date': '2015-04-01', 'value': 288307.0, 'series': 'abs'}, {'date': '2015-07-01', 'value': 297422.0, 'series': 'abs'}]


In [133]:
url = "https://api.data.gov.my/data-catalogue?limit=5"
response = requests.get(url=url)
print(type(response.json()))
print(response.json())

<class 'dict'>
{'status_code': 400, 'details': ["Query parameter 'id' is required."]}


In [134]:
import pandas as pd

# Productividad laboral trimestral por sector
URL = 'https://storage.dosm.gov.my/labour/productivity_qtr.parquet'
df_prod = pd.read_parquet(URL)
if 'date' in df_prod.columns:
    df_prod['date'] = pd.to_datetime(df_prod['date'])

print(df_prod.shape)
print(df_prod.dtypes)
print(df_prod.head(10))

(1848, 8)
series                       object
sector                       object
date                 datetime64[ns]
gdp                         float64
hours                       float64
employment                  float64
output_hour                 float64
output_employment           float64
dtype: object
  series sector       date  ...  employment  output_hour  output_employment
0    abs     p0 2015-01-01  ...     13947.0         34.7            20237.8
1    abs     p1 2015-01-01  ...      1857.0         22.8            11681.2
2    abs     p2 2015-01-01  ...        78.0        571.9           344602.6
3    abs     p3 2015-01-01  ...      2382.0         43.4            26168.3
4    abs   p3.1 2015-01-01  ...       391.0         24.6            13603.6
5    abs   p3.2 2015-01-01  ...        18.0        130.4            86944.4
6    abs   p3.3 2015-01-01  ...       222.0         10.5             5063.1
7    abs   p3.4 2015-01-01  ...       317.0         21.7            13662.5
8   

In [135]:
URL_LOOKUP = 'https://storage.dosm.gov.my/labour/productivity_lookup.parquet'
df_lookup = pd.read_parquet(URL_LOOKUP)
print(df_lookup)

          code  ...                                            desc_bm
0           p0  ...                                        Keseluruhan
1           p1  ...                                          Pertanian
2           p2  ...                       Perlombongan dan pengkuarian
3           p3  ...                                          Pembuatan
4         p3.1  ...  Minyak dan lemak daripada sayuran & haiwan dan...
5         p3.2  ...                        Minuman dan produk tembakau
6         p3.3  ...                  Produk tekstil, pakaian dan kulit
7         p3.4  ...  Produk kayu, perabot, keluaran kertas dan perc...
8         p3.5  ...         Produk petroleum, kimia, getah dan plastik
9         p3.6  ...  Produk mineral bukan logam, logam asas dan pro...
10        p3.7  ...            Produk elektrik, elektronik dan optikal
11        p3.8  ...  Peralatan pengangkutan, pembuatan lain dan pem...
12          p4  ...                                          Pembinaan
13    

In [136]:
# Filtrar sectores tecnológicos relevantes
tech_sectors = ['p3.7', 'special.04', 'special.12', 'p0']

df_tech = (
    df_prod
    .query("series == 'abs'")
    .query("sector in @tech_sectors")
    .merge(df_lookup[['code', 'desc_en']], 
           left_on='sector', right_on='code', how='left')
    [['date', 'sector', 'desc_en', 'output_hour', 'output_employment']]
    .sort_values(['sector', 'date'])
)

print(df_tech.groupby('desc_en')['date'].agg(['min', 'max']))
print(df_tech.head(15))

                                                   min        max
desc_en                                                          
Electrical, electronic and optical products 2015-01-01 2025-10-01
Overall                                     2015-01-01 2025-10-01
         date sector  desc_en  output_hour  output_employment
0  2015-01-01     p0  Overall         34.7            20237.8
2  2015-04-01     p0  Overall         35.3            20724.0
4  2015-07-01     p0  Overall         36.8            21359.5
6  2015-10-01     p0  Overall         37.2            21782.6
8  2016-01-01     p0  Overall         35.8            20880.3
10 2016-04-01     p0  Overall         36.1            21255.1
12 2016-07-01     p0  Overall         37.7            21934.2
14 2016-10-01     p0  Overall         38.5            22611.2
16 2017-01-01     p0  Overall         36.7            21645.6
18 2017-04-01     p0  Overall         37.9            22076.4
20 2017-07-01     p0  Overall         38.8            

In [137]:
fig_prod = go.Figure()

for i, (sector, label) in enumerate([('p0', 'Economía general'), 
                                      ('p3.7', 'Eléctrico y electrónico')]):
    d = df_tech[df_tech['sector'] == sector]
    fig_prod.add_trace(go.Scatter(
        x=d['date'],
        y=d['output_hour'],
        name=label,
        mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig_prod.update_layout(**BASE_LAYOUT, title=None)
fig_prod.update_yaxes(title='Valor añadido por hora trabajada (MYR)')
add_title(fig_prod,
          'Productividad laboral en Malaysia: ¿lidera la tecnología?',
          'Valor añadido por hora trabajada | 2015–2025')
add_source(fig_prod, 'OpenDOSM — Department of Statistics Malaysia')

print(df_tech.pivot_table(
    index='date', 
    columns='desc_en', 
    values='output_hour'
).round(1).to_string())

fig_prod.show()

desc_en     Electrical, electronic and optical products  Overall
date                                                            
2015-01-01                                         55.4     34.7
2015-04-01                                         49.5     35.3
2015-07-01                                         51.3     36.8
2015-10-01                                         54.6     37.2
2016-01-01                                         59.2     35.8
2016-04-01                                         53.0     36.1
2016-07-01                                         55.5     37.7
2016-10-01                                         57.3     38.5
2017-01-01                                         61.1     36.7
2017-04-01                                         58.4     37.9
2017-07-01                                         57.8     38.8
2017-10-01                                         59.8     40.2
2018-01-01                                         64.7     38.2
2018-04-01               

In [138]:
fig_prod = go.Figure()

for i, (sector, label) in enumerate([('p0', 'Economía general'), 
                                      ('p3.7', 'Eléctrico y electrónico')]):
    d = df_tech[df_tech['sector'] == sector]
    fig_prod.add_trace(go.Scatter(
        x=d['date'],
        y=d['output_hour'],
        name=label,
        mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig_prod.update_layout(**BASE_LAYOUT, title=None)
fig_prod.update_yaxes(title='Valor añadido por hora trabajada (MYR)')
add_title(fig_prod,
          'Productividad laboral en Malaysia: ¿lidera la tecnología?',
          'Valor añadido por hora trabajada | 2015–2025')
add_source(fig_prod, 'OpenDOSM — Department of Statistics Malaysia')
fig_prod.show()

In [139]:
# Crecimiento YoY
df_tech_yoy = (
    df_tech
    .sort_values(['sector', 'date'])
    .assign(yoy=lambda x: x.groupby('sector')['output_hour']
            .pct_change(4) * 100)
    .dropna(subset=['yoy'])
)

fig_yoy = go.Figure()

for i, (sector, label) in enumerate([('p0', 'Economía general'),
                                      ('p3.7', 'Eléctrico y electrónico')]):
    d = df_tech_yoy[df_tech_yoy['sector'] == sector]
    fig_yoy.add_trace(go.Scatter(
        x=d['date'],
        y=d['yoy'],
        name=label,
        mode='lines',
        line=dict(color=COLORS[i], width=2)
    ))

fig_yoy.add_hline(y=0, line=dict(color='#2c3e50', width=1, dash='dash'))

fig_yoy.update_layout(**BASE_LAYOUT, title=None)
fig_yoy.update_yaxes(title='Crecimiento YoY (%)', ticksuffix='%')
add_title(fig_yoy,
          'Crecimiento de productividad laboral en Malaysia',
          'Variación anual | 2016–2025')
add_source(fig_yoy, 'OpenDOSM — Department of Statistics Malaysia')

print(df_tech_yoy.pivot_table(
    index='date',
    columns='desc_en',
    values='yoy'
).round(2).to_string())

fig_yoy.show()

desc_en     Electrical, electronic and optical products  Overall
date                                                            
2016-01-01                                         6.86     3.17
2016-04-01                                         7.07     2.27
2016-07-01                                         8.19     2.45
2016-10-01                                         4.95     3.49
2017-01-01                                         3.21     2.51
2017-04-01                                        10.19     4.99
2017-07-01                                         4.14     2.92
2017-10-01                                         4.36     4.42
2018-01-01                                         5.89     4.09
2018-04-01                                         5.48     3.69
2018-07-01                                         1.90     3.09
2018-10-01                                         5.02     2.99
2019-01-01                                         2.63     2.62
2019-04-01               

In [140]:
# Brecha de productividad: sector tecnológico vs economía general
df_gap = (
    df_tech
    .query("series == 'abs'")
    .pivot_table(index='date', columns='sector', values='output_hour')
    .reset_index()
    .assign(gap=lambda x: x['p3.7'] - x['p0'])
)

fig_gap = go.Figure()

# Área bajo la curva para mayor impacto visual
fig_gap.add_trace(go.Scatter(
    x=df_gap['date'],
    y=df_gap['gap'],
    mode='lines',
    fill='tozeroy',
    fillcolor='rgba(44, 62, 80, 0.15)',
    line=dict(color='#2c3e50', width=2),
    showlegend=False
))

fig_gap.add_hline(y=0, line=dict(color='#e74c3c', width=1, dash='dash'))

fig_gap.update_layout(**BASE_LAYOUT, title=None)
fig_gap.update_yaxes(title='Diferencia en MYR por hora trabajada')
add_title(fig_gap,
          'Brecha de productividad: tecnología vs economía general',
          'Sector eléctrico-electrónico menos media nacional | Malaysia 2015–2025')
add_source(fig_gap, 'OpenDOSM — Department of Statistics Malaysia')

print(df_gap[['date', 'gap']].round(2).to_string())
fig_gap.show()

UndefinedVariableError: name 'series' is not defined

In [141]:
df_gap = (
    df_tech
    .pivot_table(index='date', columns='sector', values='output_hour')
    .reset_index()
    .assign(gap=lambda x: x['p3.7'] - x['p0'])
)

fig_gap = go.Figure()

fig_gap.add_trace(go.Scatter(
    x=df_gap['date'],
    y=df_gap['gap'],
    mode='lines',
    fill='tozeroy',
    fillcolor='rgba(44, 62, 80, 0.15)',
    line=dict(color='#2c3e50', width=2),
    showlegend=False
))

fig_gap.add_hline(y=0, line=dict(color='#e74c3c', width=1, dash='dash'))

fig_gap.update_layout(**BASE_LAYOUT, title=None)
fig_gap.update_yaxes(title='Diferencia en MYR por hora trabajada')
add_title(fig_gap,
          'Brecha de productividad: tecnología vs economía general',
          'Sector eléctrico-electrónico menos media nacional | Malaysia 2015–2025')
add_source(fig_gap, 'OpenDOSM — Department of Statistics Malaysia')

print(df_gap[['date', 'gap']].round(2).to_string())
fig_gap.show()

sector       date   gap
0      2015-01-01  20.7
1      2015-04-01  14.2
2      2015-07-01  14.5
3      2015-10-01  17.4
4      2016-01-01  23.4
5      2016-04-01  16.9
6      2016-07-01  17.8
7      2016-10-01  18.8
8      2017-01-01  24.4
9      2017-04-01  20.5
10     2017-07-01  19.0
11     2017-10-01  19.6
12     2018-01-01  26.5
13     2018-04-01  22.3
14     2018-07-01  18.9
15     2018-10-01  21.4
16     2019-01-01  27.2
17     2019-04-01  22.1
18     2019-07-01  20.1
19     2019-10-01  22.1
20     2020-01-01  29.5
21     2020-04-01  38.1
22     2020-07-01  27.5
23     2020-10-01  28.6
24     2021-01-01  37.1
25     2021-04-01  34.8
26     2021-07-01  34.2
27     2021-10-01  35.0
28     2022-01-01  43.2
29     2022-04-01  36.9
30     2022-07-01  36.8
31     2022-10-01  37.0
32     2023-01-01  41.5
33     2023-04-01  31.6
34     2023-07-01  31.1
35     2023-10-01  30.1
36     2024-01-01  38.2
37     2024-04-01  33.5
38     2024-07-01  34.1
39     2024-10-01  34.2
40     2025-01-0

In [142]:
import requests

url = "https://hai.stanford.edu/assets/files/hai_ai-index-report-2025_chapter4_final.pdf"
response = requests.get(url, timeout=30)
print(f"Status: {response.status_code}")
print(f"Size: {len(response.content)} bytes")

# Guardar el PDF
with open('/home/claude/hai_chapter4.pdf', 'wb') as f:
    f.write(response.content)
print("PDF guardado")

Status: 200
Size: 5303819 bytes


FileNotFoundError: [Errno 2] No such file or directory: '/home/claude/hai_chapter4.pdf'

In [143]:
import requests

url = "https://hai.stanford.edu/assets/files/hai_ai-index-report-2025_chapter4_final.pdf"
response = requests.get(url, timeout=30)
print(f"Status: {response.status_code}")
print(f"Size: {len(response.content)} bytes")

# Guardar en directorio local
with open('hai_chapter4.pdf', 'wb') as f:
    f.write(response.content)
print("PDF guardado")

Status: 200
Size: 5303819 bytes
PDF guardado


In [144]:
import subprocess
result = subprocess.run(
    ['pdftotext', 'hai_chapter4.pdf', '-'],
    capture_output=True, text=True
)
text = result.stdout
# Buscar sección de adopción por país
idx = text.find('adoption')
print(text[idx:idx+3000])

FileNotFoundError: [Errno 2] No such file or directory: 'pdftotext'

In [146]:
import pypdf

reader = pypdf.PdfReader('hai_chapter4.pdf')
print(f"Páginas: {len(reader.pages)}")

# Extraer texto de las primeras 20 páginas buscando datos de adopción
text = ''
for i in range(min(20, len(reader.pages))):
    text += reader.pages[i].extract_text()

idx = text.lower().find('adoption')
print(text[idx:idx+3000])

Páginas: 77
adoption toward systematic integration. 
Investment patterns reflect a growing sophistication in the AI landscape, with funding 
increasingly directed toward specialized applications in enterprise automation and 
industry-specific solutions.
This chapter examines AI-related economic trends using data from Lightcast, LinkedIn, 
Quid, McKinsey and the International Federation of Robotics (IFR). It begins by analyzing 
AI-related occupations, covering labor demand, hiring trends, skill penetration, and 
talent availability. The chapter then explores corporate investment in AI, including 
a section focused specifically on generative AI. Finally, it assesses AI’s productivity 
impact as well as robot installations across various sectors.
Overview
CHAPTER 4: 
Economy
Artificial Intelligence
Index Report 20254
Artificial Intelligence
Index Report 2025Chapter 4 Preview
Chapter Highlights
1. Global private AI investment hits record high with 26% growth. Corporate AI investment reach

In [147]:
# Buscar datos específicos de EAP y adopción por país
keywords = ['Greater China', 'Asia', 'Malaysia', 'Vietnam', 'Indonesia', 
            'Philippines', 'adoption by country', 'regional']

for kw in keywords:
    idx = text.lower().find(kw.lower())
    if idx > 0:
        print(f"\n=== {kw} ===")
        print(text[idx:idx+500])
        print("...")


=== Greater China ===
Greater China gaining ground.  While North America 
maintains its leadership in organizations’ use of AI, Greater China demonstrated one of the most significant year-over-year 
growth rates, with a 27 percentage point increase in organizational AI use. Europe followed with a 23 percentage point 
increase, suggesting a rapidly evolving global AI landscape and intensifying international competition in AI implementation.
CHAPTER 4: 
Economy
Artificial Intelligence
Index Report 20255
Artificial Inte
...

=== Indonesia ===
Indonesia, India, and South 
Korea, included in the sample, have LinkedIn covering a 
lower portion of the labor force, so insights drawn about 
these countries should be interpreted with particular caution.
Figure 4.2.13 reports the relative AI hiring rate year-over-year 
ratio by geographic area. The overall hiring rate is computed 
as the percentage of LinkedIn members who added a new 
employer in the same period the job began, divided by the 
to

In [148]:
# Extraer páginas 20-40 donde están los datos de adopción corporativa
text2 = ''
for i in range(20, 50):
    text2 += reader.pages[i].extract_text()

# Buscar datos de adopción por región
idx = text2.lower().find('greater china')
print(text2[idx:idx+2000])

Greater China, where 
organizations’ reported use grew by 27 percentage points. 
North America remains the leader in use of AI (82%), but 
only by a small margin. Europe also experienced a significant 
increase in AI usage rates, growing by 23 percentage points 
to 80% since 2023.
78%
72%
80%
82%
75%
77%
55%
58%
57%
61%
48%
49%
0% 10% 20% 30% 40% 50% 60% 70% 80% 90% 100%
Developing markets
(incl. India,
Central/South America,
MENA)
Greater China
(incl. Hong Kong,
Taiwan, Macau)
North America
Europe
Asia-Pacic
All geographies
2024
2023
% of respondents
AI use by organizations in the world, 2023 vs. 2024
Source: McKinsey & Company Survey, 2024 | Chart: 2025 AI Index report
Figure 4.4.4
4.4 Corporate Activity
Chapter 4: Economy


In [149]:
# Datos de adopción organizacional IA por región
adoption_data = {
    'Region': [
        'Asia-Pacífico', 'Europa', 'North America', 
        'Greater China', 'Developing Markets', 'Global'
    ],
    '2023': [61, 57, 58, 48, 49, 55],
    '2024': [77, 80, 82, 75, 72, 78]
}

df_adoption = pd.DataFrame(adoption_data)
df_adoption['change'] = df_adoption['2024'] - df_adoption['2023']

fig_adopt = go.Figure()

fig_adopt.add_trace(go.Bar(
    x=df_adoption['Region'],
    y=df_adoption['2023'],
    name='2023',
    marker_color='rgba(44, 62, 80, 0.4)'
))

fig_adopt.add_trace(go.Bar(
    x=df_adoption['Region'],
    y=df_adoption['2024'],
    name='2024',
    marker_color='#2c3e50'
))

fig_adopt.update_layout(
    **BASE_LAYOUT,
    title=None,
    barmode='group'
)
fig_adopt.update_yaxes(title='% organizaciones que usan IA', ticksuffix='%')
add_title(fig_adopt,
          'Adopción organizacional de IA por región',
          '2023 vs 2024 | % de organizaciones que reportan uso de IA')
add_source(fig_adopt, 'McKinsey & Company Survey 2024 — Stanford HAI AI Index 2025')
fig_adopt.show()

In [150]:
# Extraer páginas 40-60
text3 = ''
for i in range(40, 60):
    text3 += reader.pages[i].extract_text()

# Buscar datos de productividad e impacto económico
keywords = ['productivity', 'revenue', 'cost', 'impact', 'Asia', 'EAP']

for kw in keywords:
    idx = text3.lower().find(kw.lower())
    if idx > 0:
        print(f"\n=== {kw} ===")
        print(text3[idx:idx+800])
        print("...")


=== productivity ===
productivity 
potential. While early adoption showed promise, quantifying 
AI’s impact remained challenging until 2023, when the first 
wave of rigorous studies emerged. In 2024, a substantial 
body of empirical research established clear patterns of AI’s 
workplace effects across multiple domains and contexts. This 
section analyzes productivity impact data from five major 
academic studies, which together represent the first large-
scale empirical investigation of AI’s workplace effects. The 
research, encompassing over 200,000 professionals across 
multiple industries and contexts, reveals consistent productivity 
gains ranging from 10% to 45%, with particularly strong effects 
in technical, customer support, and creative tasks. These 
studies employed diverse methodologies, including n
...

=== revenue ===
revenue increases where they have started using AI, but 
most commonly at low levels (Figure 4.4.3). The areas where 
respondents most frequently reported t

In [151]:
# Extraer páginas 1-20 — datos de empleo y contratación IA
text4 = ''
for i in range(1, 20):
    text4 += reader.pages[i].extract_text()

keywords = ['hiring', 'talent', 'skills', 'wage', 'salary', 
            'Southeast Asia', 'Malaysia', 'Vietnam', 'Philippines']

for kw in keywords:
    idx = text4.lower().find(kw.lower())
    if idx > 0:
        print(f"\n=== {kw} ===")
        print(text4[idx:idx+800])
        print("...")


=== hiring ===
Hiring 19
AI Skill Penetration 21
AI Talent 23
Highlight: Measuring AI’s Current  
Economic Integration 29
4.3 Investment 33
Corporate Investment 33
Startup Activity 34
 Global Trends 34
 Regional Comparison by  
 Funding Amount 38
 Regional Comparison by  
 Newly Funded AI Companies 42
 Focus Area Analysis 45
4.4 Corporate Activity 47
Industry Usage 47
 Use of AI Capabilities 47
 Deployment of Generative  
 AI Capabilities  51
AI’s Labor Impact 54
4.5 Robot Deployments 59
Aggregate Trends 59
 Industrial Robots: Traditional  
 vs. Collaborative Robots 61
By Geographic Area 62
 Country-Level Data on Service  
 Robotics 66
Appendix 67
Chapter 4: Economy
Artificial Intelligence
Index Report 2025
ACCESS THE PUBLIC DATA3
Artificial Intelligence
Index Report 2025Chapter 4 Preview
The economic im
...

=== talent ===
Talent 23
Highlight: Measuring AI’s Current  
Economic Integration 29
4.3 Investment 33
Corporate Investment 33
Startup Activity 34
 Global Trends 34
 Regional Com

In [152]:
# Contraste BM vs McKinsey/Stanford
df_contrast = pd.DataFrame({
    'Fuente': [
        'Banco Mundial\n(subsidiarias multinacionales)',
        'McKinsey/Stanford HAI\n(organizaciones en general)'
    ],
    'Adopcion': [15, 77],  # 15% media del rango 13-17% del BM
    'Color': ['#e74c3c', '#2c3e50']
})

fig_contrast = go.Figure()

fig_contrast.add_trace(go.Bar(
    x=df_contrast['Fuente'],
    y=df_contrast['Adopcion'],
    marker_color=df_contrast['Color'],
    text=df_contrast['Adopcion'].astype(str) + '%',
    textposition='outside',
    textfont=dict(size=14, family='Georgia, serif', color='#2c3e50'),
    showlegend=False,
    width=0.4
))

fig_contrast.update_layout(**BASE_LAYOUT, title=None)
fig_contrast.update_yaxes(
    title='% adopción de IA',
    ticksuffix='%',
    range=[0, 100]
)
add_title(fig_contrast,
          '¿Quién tiene razón sobre la adopción de IA en Asia-Pacífico?',
          'Banco Mundial (2026) vs McKinsey/Stanford HAI (2025) | Asia-Pacífico')
add_source(fig_contrast,
           'Banco Mundial EAP Economic Update 2026 | Stanford HAI AI Index 2025')
fig_contrast.show()

In [153]:
# Buscar datos completos de AI job postings por país
idx = text4.find('Singapore (3.2%)')
print(text4[idx:idx+2000])

Singapore (3.2%), 
Luxembourg (2%), and Hong Kong (1.9%) led in this metric. 
In 2023, AI-related jobs accounted for 1.4% of all American 
job postings. In 2024, that number increased to 1.8%. Most 
countries saw an increase from 2023 to 2024 in the share of 
job postings requiring AI skills.
2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024
0.00%
1.00%
2.00%
3.00%
4.00%
5.00%
AI job postings (% of all job postings)
1.25%, Netherlands
1.26%, United Kingdom
1.31%, Sweden
1.31%, Belgium
1.37%, Switzerland
1.41%, Canada
1.72%, United Arab Emirates
1.79%, United States
1.89%, Hong Kong
1.99%, Luxembourg
3.27%, Singapore
AI job postings (% of all job postings) by select geographic areas, 2014– 24 (part 1)
Source: Lightcast, 2024 | Chart: 2025 AI Index report
Figure 4.2.1
4.2 Jobs
Chapter 4: Economy11
Artificial Intelligence
Index Report 2025Chapter 4 Preview
4.2 Jobs
Chapter 4: Economy
2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024
0.00%
1.00%
2.00%
3.00%
4.00%
5.00%
AI job pos

In [155]:
# AI job postings por país 2024
df_jobs = pd.DataFrame({
    'País': [
        'Singapore', 'Luxembourg', 'Hong Kong', 'UAE',
        'United States', 'Canada', 'Switzerland',
        'Belgium', 'Sweden', 'UK', 'Netherlands',
        'Germany', 'Australia', 'France', 'Austria',
        'Italy', 'Mexico', 'Chile', 'New Zealand', 'Croatia'
    ],
    'ai_jobs_pct': [
        3.27, 1.99, 1.89, 1.72,
        1.79, 1.41, 1.37,
        1.31, 1.31, 1.26, 1.25,
        1.15, 1.14, 1.10, 1.06,
        0.87, 0.73, 0.65, 0.55, 0.13
    ],
    'region': [
        'EAP', 'Europa', 'EAP', 'MENA',
        'Norte América', 'Norte América', 'Europa',
        'Europa', 'Europa', 'Europa', 'Europa',
        'Europa', 'Oceanía', 'Europa', 'Europa',
        'Europa', 'LATAM', 'LATAM', 'Oceanía', 'Europa'
    ]
})

df_jobs = df_jobs.sort_values('ai_jobs_pct', ascending=True)

# Colores por región
color_map = {
    'EAP': '#e74c3c',
    'Norte América': '#2c3e50',
    'Europa': '#95a5a6',
    'MENA': '#e67e22',
    'LATAM': '#27ae60',
    'Oceanía': '#3498db'
}

fig_jobs = go.Figure()

fig_jobs.add_trace(go.Bar(
    x=df_jobs['ai_jobs_pct'],
    y=df_jobs['País'],
    orientation='h',
    marker_color=[color_map[r] for r in df_jobs['region']],
    showlegend=False
))

# Añadir leyenda manual
for region, color in color_map.items():
    fig_jobs.add_trace(go.Bar(
        x=[None], y=[None],
        name=region,
        marker_color=color,
        showlegend=True
    ))

fig_jobs.update_layout(
    **BASE_LAYOUT,
    title=None,
    height=600,
    barmode='overlay'
)
fig_jobs.update_xaxes(title='% ofertas de empleo que requieren habilidades de IA', ticksuffix='%')
add_title(fig_jobs,
          'Demanda laboral de IA: EAP ausente del mapa global',
          '% de ofertas de empleo que requieren habilidades de IA | 2024')
add_source(fig_jobs, 'Lightcast 2024 — Stanford HAI AI Index 2025')
fig_jobs.show()

In [156]:
fig_contrast = go.Figure()

fig_contrast.add_trace(go.Bar(
    x=['Banco Mundial\n(subsidiarias multinacionales)',
       'McKinsey/Stanford HAI\n(organizaciones en general)'],
    y=[15, 77],
    marker_color=['#e74c3c', '#2c3e50'],
    text=['13–17%', '77%'],
    textposition='outside',
    textfont=dict(size=14, family='Georgia, serif', color='#2c3e50'),
    showlegend=False,
    width=0.4
))

fig_contrast.update_layout(**BASE_LAYOUT, title=None)
fig_contrast.update_yaxes(
    title='% adopción de IA',
    ticksuffix='%',
    range=[0, 100]
)
add_title(fig_contrast,
          '¿Quién tiene razón sobre la adopción de IA en Asia-Pacífico?',
          'Banco Mundial (2026) vs McKinsey/Stanford HAI (2025) | Asia-Pacífico')
add_source(fig_contrast,
           'Banco Mundial EAP Economic Update 2026 | Stanford HAI AI Index 2025')
fig_contrast.show()

In [157]:
# Impacto financiero de IA por función de negocio
df_impact = pd.DataFrame({
    'Función': [
        'Marketing y ventas',
        'Supply chain',
        'Service operations',
        'Software engineering',
        'IT',
        'Product development',
        'HR',
        'Risk y legal'
    ],
    'cost_savings': [25, 43, 49, 41, 37, 23, 37, 34],
    'revenue_gains': [71, 63, 57, 34, 28, 43, 20, 15]
})

df_impact = df_impact.sort_values('revenue_gains', ascending=True)

fig_impact = go.Figure()

fig_impact.add_trace(go.Bar(
    x=df_impact['cost_savings'],
    y=df_impact['Función'],
    orientation='h',
    name='Ahorro de costes',
    marker_color='rgba(44, 62, 80, 0.5)',
))

fig_impact.add_trace(go.Bar(
    x=df_impact['revenue_gains'],
    y=df_impact['Función'],
    orientation='h',
    name='Incremento de ingresos',
    marker_color='#2c3e50',
))

fig_impact.update_layout(
    **BASE_LAYOUT,
    title=None,
    barmode='overlay',
    height=500
)
fig_impact.update_xaxes(title='% organizaciones que reportan impacto', ticksuffix='%')
add_title(fig_impact,
          'Impacto financiero de la IA por función de negocio',
          '% organizaciones que reportan beneficios | Global 2024')
add_source(fig_impact, 'McKinsey & Company Survey 2024 — Stanford HAI AI Index 2025')
fig_impact.show()

### 5.4 Adopción alta, impacto modesto

El gráfico de impacto financiero revela una tensión estructural que 
el informe del Banco Mundial no aborda: incluso en las organizaciones 
que ya han adoptado IA, los beneficios financieros son predominantemente 
modestos.

**Marketing y ventas lidera en ingresos pero no en costes.** El 71% 
de organizaciones que usan IA en marketing reportan ganancias de 
ingresos — la cifra más alta del panel — pero solo el 25% reporta 
ahorro de costes en esa misma función. La IA en marketing es 
principalmente una herramienta de crecimiento, no de eficiencia.

**Service operations y supply chain lideran en ahorro de costes.** 
El 49% y 43% respectivamente reportan reducciones de coste — 
consistente con la naturaleza repetitiva y automatizable de estas 
funciones. Son los casos de uso más maduros y mejor documentados 
de IA empresarial.

**La advertencia clave.** El informe Stanford HAI especifica que 
la mayoría de organizaciones que reportan impacto financiero lo 
sitúan en niveles bajos — reducciones de coste inferiores al 10% 
e incrementos de ingresos inferiores al 5%. La adopción es amplia 
pero la profundidad del impacto es todavía limitada.

Para los países de EAP esto tiene una implicación directa: acelerar 
la adopción de IA sin construir capacidad para extraer valor 
productivo real de esa adopción reproduce exactamente el patrón 
que hemos documentado en las exportaciones tecnológicas — alta 
intensidad superficial, bajo valor añadido doméstico.